# アノテーション

アノテーションでは、11_record_cameraで撮影した走行データにアノテーションを実施し、転移学習をおこないます。

収集した走行データを用いて、アノテーションをし、データセットを作成します。

### ボードの識別

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO          # もう一度ロード
    BOARD_NAME = "JETSON_ORIN_NANO"     # 確定

# ---------- 2. ボード別定義 ----------
mode_descriptions = {
    "JETSON_NX":       ["15W_2CORE", "15W_4CORE", "15W_6CORE", "10W_2CORE", "10W_4CORE"],
    "JETSON_XAVIER":   ["MAXN", "MODE_10W", "MODE_15W", "MODE_30W"],
    "JETSON_NANO":     ["MAXN", "5W"],
    "JETSON_ORIN":     ["MAXN", "MODE_15W", "MODE_30W", "MODE_40W"],
    "JETSON_ORIN_NANO":["MODE_15W", "MODE_25W", "MODE_MAX"]
}

product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

# (I2C バス番号, 初期 Power モードインデックス)
board_settings = {
    "JETSON_NX":        (8, 3),
    "JETSON_XAVIER":    (8, 2),
    "JETSON_NANO":      (1, 0),
    "JETSON_ORIN":      (7, 0),
    "JETSON_ORIN_NANO": (7, 2)
}

# ---------- 3. パラメータ取得 ----------
i2c_busnum, power_mode = board_settings.get(BOARD_NAME, (None, None))
mode_list       = mode_descriptions.get(BOARD_NAME, [])
product_name    = product_names.get(BOARD_NAME, "未知のボード")

# ---------- 4. 出力 ----------
if i2c_busnum is not None and 0 <= power_mode < len(mode_list):
    mode_str = mode_list[power_mode]
    print("------------------------------------------------------------")
    print(f"{product_name} を認識: I2C バス番号 = {i2c_busnum}, "
          f"Power モード = {mode_str} ({power_mode})")
    print("------------------------------------------------------------")
else:
    raise RuntimeError(f"未対応の Jetson モデル、または Power モード定義不足: {BOARD_NAME}")

In [ ]:
!echo "jetson" | sudo -S nvpmodel -m $power_mode

In [ ]:
!echo "jetson" | sudo -S nvpmodel -q

In [ ]:
!echo "jetson" | sudo -S jetson_clocks

走行データはcameraフォルダに録画されています。今度は、cameraフォルダのデータにアノテーションをおこないます。

In [ ]:
import os
import re
import threading
import time
from datetime import datetime, timezone
from functools import partial
from types import SimpleNamespace

import cv2
import ipywidgets
import matplotlib.image as mpimg
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
import re
import torch
import torchvision
import torchvision.transforms as transforms
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label
from jetcam.utils import bgr8_to_jpeg
from jupyter_clickable_image_widget import ClickableImageWidget

from fabo import asset_root
from fabo.annotation import draw_grids
from utils import preprocess
from xy_dataset import XYDataset

In [ ]:
IMG_WIDTH = 224
IMG_HEIGHT = 224

SLEEP = [50,100,200,300,400,500]
SKIP = [1,2,3,4,5]

LOAD_CATEGORIES = ['xy','speed']
SAVE_CATEGORIES = ['xy','speed']

LOAD_TASK = ['camera','dataset','interactive']
SAVE_TASK = ['dataset']

check_flag = False
running = False
sleep_time = 30

def get_dirs(path):
    files = os.listdir(path)
    dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]
    dirs = [f for f in dirs if f != ".ipynb_checkpoints"]
    dirs = sorted(dirs)

    return dirs

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

datasets = {}


In [ ]:
l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0
def write_log(msg):
    global process_widget, process_no
    process_no = process_no + 1
    process_widget.value = str(process_no) + ": " + msg + "\n" + process_widget.value

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')

sleep_dropdown = ipywidgets.Dropdown(options=SLEEP, description='sleep(ms)', index=1)
skip_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)
movie_skip_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)

picture_widget_extra_height = 14  # 速度バー描画用の追加スペース
picture_widget = ClickableImageWidget(
    width=224, height=224 + picture_widget_extra_height,
    layout=ipywidgets.Layout(width="224px", height=f"{224 + picture_widget_extra_height}px"))
picture_widget.format = "jpeg"

no_widget = ipywidgets.IntText(description='no')
y_widget = ipywidgets.IntText(description='data y')
speed_widget = ipywidgets.IntText(description='data speed')
model_widget = ipywidgets.Text(description='model')
model_widget.value = "model.pth"
load_model_button = ipywidgets.Button(description='load model')

class AnnotationWidget:
    def __init__(self, title, value_types, save_func, migrate_func):
        self.title = ipywidgets.Label(title)
        self.value_types = value_types
        self.migration_dataset_title_label = ipywidgets.Label("移行先データセット")
        self.migration_datasets_dropdown = ipywidgets.Dropdown(layout=ipywidgets.Layout(width="120px"))
        self.migration_datasets_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
        self.migration_button = ipywidgets.Button(description="データセット移行", button_style="warning")
        self.delete_button = ipywidgets.Button(description="アノテーション削除", button_style="danger")

        self.from_inference = self._generate_widgets_per_annotation_source("推論", save_func)
        self.from_input = self._generate_widgets_per_annotation_source("camera", save_func)
        self.from_annotation = self._generate_widgets_per_annotation_source("dataset", save_func)

        self.migration_datasets_refresh_button.on_click(self.on_migration_datasets_refresh)
        self.migration_button.on_click(partial(migrate_func, migration_datasets_dropdown=self.migration_datasets_dropdown))
        self.delete_button.on_click(self.on_delete)

        self.on_migration_datasets_refresh(None)

    def on_migration_datasets_refresh(self, button):
        global save_task_widget
        path = os.path.join(asset_root(), save_task_widget.value)
        try:
            os.makedirs(path, exist_ok=True)
            dirs = get_dirs(path)
            self.migration_datasets_dropdown.options = dirs
            # 保存先データセットは明示的に指定させる
            # if len(self.migration_datasets_dropdown.options) > 0 and self.migration_datasets_dropdown.value not in self.migration_datasets_dropdown.options:
            #     self.migration_datasets_dropdown.index = 0
        except:
            write_log(path + "が存在していません。")
            self.migration_datasets_dropdown.options = ['']

    def on_delete(self, button):
        global save_task_widget, img_filename
        name = save_datasets_widget.value
        dataset_path = os.path.join(asset_root(), save_task_widget.value, name)
        for value_type in self.value_types:
            pop_from_parquet(dataset_path, img_filename, value_type)

        file_count()

        update_image(None)

    def _generate_widgets_per_annotation_source(self, title, save_func, width="80px"):
        common_layout = ipywidgets.Layout(width=width)
        ns = SimpleNamespace()
        ns.slider = ipywidgets.IntSlider(description=title, min=0, max=224, step=1, value=0, orientation='vertical', layout=ipywidgets.Layout(border="solid transparent"))
        ns.add_button = ipywidgets.Button(description="追加", layout=common_layout)
        ns.offset_label = ipywidgets.Label("調整", tooltip="値に恒常的なズレがある場合、この値で調整できます", layout=common_layout)
        ns.offset_form = ipywidgets.IntText(min=-112, max=112, step=1, value=0, layout=common_layout)

        ns.add_button.on_click(partial(save_func, slider=ns.slider))

        return ns

    @property
    def widget(self):
        return ipywidgets.VBox([
            self.title,
            ipywidgets.HBox([
                ipywidgets.VBox(list(self.from_inference.__dict__.values())),
                ipywidgets.VBox(list(self.from_input.__dict__.values())),
                ipywidgets.VBox(list(self.from_annotation.__dict__.values())),
            ]),
            separator,
            self.migration_dataset_title_label,
            ipywidgets.HBox([
                self.migration_datasets_dropdown,
                self.migration_datasets_refresh_button,
            ]),
            self.migration_button,
            separator,
            self.delete_button,
        ])


In [ ]:
from packaging import version

torchvision_version = version.parse(torchvision.__version__)

device = torch.device('cuda')
output_dim = 2 * len(LOAD_CATEGORIES)
model = None
model_metadata = {}

In [ ]:
from pathlib import Path

import yaml

from fabo import asset_root


def load_model(c):
    global model, model_metadata

    def new_model_metadata():
        return {
            "model": None,
            "model_type": "ResNet18",
            "torchvision_version": str(torchvision_version),
        }

    def load_model_metadata(_model_name):
        p = Path(_model_name)
        load_model_metadata_path = p.parent / (p.stem + ".yaml")
        if load_model_metadata_path.exists():
            _model_metadata = None
            try:
                with open(load_model_metadata_path, "r") as f:
                    _model_metadata = yaml.safe_load(f)
            except Exception as e:
                write_log(f"{load_model_metadata_path} の読み込みに失敗しました: {e}")
            finally:
                if _model_metadata is None:
                    _model_metadata = {}
        else:
            _model_metadata = {}
        _model_metadata |= {
            "model": str(Path(_model_name).relative_to(asset_root())),
            "model_type": "ResNet18",
            "torchvision_version": str(torchvision_version),
        }
        return _model_metadata

    model_name = load_model_widget.value
    if torchvision_version >= version.parse("0.13"):
        # torchvision 0.13以降の場合
        from torchvision.models.resnet import ResNet18_Weights, resnet18

        if model_name == "[new]":
            # 新しい重みを使ってモデルをロード
            default_weights = ResNet18_Weights.DEFAULT
            model = resnet18(weights=default_weights)
            model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
            model_metadata = new_model_metadata()
            write_log("[new]が選択されたのでresnet18の最新の重みから始めます(torchvision 0.13以降)。")
        else:
            model = resnet18(weights=None)  # pretrained=Falseの代わり
            model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
            state_dict = torch.load(model_name, weights_only=True)
            model.load_state_dict(state_dict)
            model_metadata = load_model_metadata(model_name)
            write_log(model_name + "のモデルを読込ました(torchvision 0.13以降)。")

    else:
        # torchvision 0.13より前の場合
        if model_name == "[new]":
            model = torchvision.models.resnet18(pretrained=True)
            model.fc = torch.nn.Linear(512, output_dim)
            model_metadata = new_model_metadata()
            write_log("[new]が選択されたのでresnet18のpretrainedから始めます。")
        else:
            model = torchvision.models.resnet18(pretrained=False)
            model.fc = torch.nn.Linear(512, output_dim)
            model.load_state_dict(torch.load(model_name))
            model_metadata = load_model_metadata(model_name)
            write_log(model_name + "のモデルを読込ました。")

    model.eval()
    model = model.to(device)
    get_jetson_nano_memory_usage()


def save_model(c):
    global model_metadata
    path = os.path.join(asset_root(), "model")
    os.makedirs(path, exist_ok=True)
    save_model_path = os.path.join(path, save_model_name_widget.value)
    if save_best_model_checkbox.value:
        import shutil
        save_model_best_path = os.path.join(path, "best_model.pth")
        shutil.copy2(save_model_best_path, save_model_path)
        # Bestモデルを読み込む
        load_model_widget.value = save_model_path
        load_model(None)
    else:
        torch.save(model.state_dict(), save_model_path)

    p = Path(save_model_path)
    model_metadata |= {
        "model": str(p.relative_to(asset_root())),
        "save_best_model": save_best_model_checkbox.value,
    }
    save_model_metadata_path = p.parent / (p.stem + ".yaml")
    with open(save_model_metadata_path, "w") as f:
        yaml.dump(model_metadata, f)

    write_log(save_model_path + "に保存しました。")


In [ ]:
import time

from fabo import asset_root, format_epoch

BATCH_SIZE = 8

epochs_widget = ipywidgets.IntText(description='epochs', value=1)
eval_button = ipywidgets.Button(description='evaluate')
train_button = ipywidgets.Button(description='train')
loss_widget = ipywidgets.FloatText(description='loss', disabled=True)
best_loss_form = ipywidgets.Text(description='best_loss', disabled=True, value="inf")
reset_best_loss_button = ipywidgets.Button(description="reset best_loss")
progress_widget = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')

best_loss = float('inf')

def train_eval(is_training):
    global best_loss, model, model_metadata
    
    save_dataset_name = save_datasets_widget.value
    save_dataset_path = os.path.join(asset_root(), save_task_widget.value, save_dataset_name)
    dataset = XYDataset(save_dataset_path, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)

    # ベースモデル情報を学習前モデル情報に設定し、モデル情報を初期化する
    model_metadata |= {
        "base_model": model_metadata["model"],
        "model": None,
    }
    # 各種学習構成を設定
    model_metadata |= {
        "batch_size": BATCH_SIZE,
        "dataset": str(Path(save_dataset_path).relative_to(asset_root())),
        "dataset_categories": {x: {"samples_size": dataset.get_count(x)} for x in SAVE_CATEGORIES},
        "epochs": epochs_widget.value,
        "sample_size": len(dataset),
    }

    write_log(f"-------------------------")
    write_log(f"学習を開始します。")
    write_log(f"データセット: {dataset.directory}")
    write_log(f"カテゴリ別データ数:")
    for x in SAVE_CATEGORIES:
        write_log(f"  {x} データ数: {dataset.get_count(x)}")
    write_log(f"-------------------------")

    # 有効なデータのインデックスを確認
    valid_indices = []
    for i in range(len(dataset)):
        try:
            item = dataset[i]
            if item is not None and item[0] is not None:
                valid_indices.append(i)
        except Exception as e:
            pass
        
    # 有効なインデックスを持つサブセットを作成
    valid_dataset = torch.utils.data.Subset(dataset, valid_indices)

    try:
        optimizer = torch.optim.Adam(model.parameters())

        train_loader = torch.utils.data.DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        train_button.disabled = True
        eval_button.disabled = True
        time.sleep(1)

        if is_training:
            model = model.train()
        else:
            model = model.eval()
        epoch_count = 0
        global_start_time = time.time()

        while epochs_widget.value > 0:
            epoch_start_time = time.time()  # エポック開始時間を記録
            epoch_count += 1
            i = 0
            sum_loss = 0.0
            error_count = 0.0
            for images, category_idx, xy in iter(train_loader):
                if images is None or xy is None:
                    print("Warning: None type data found at index", i)
                    continue
                images = images.to(device)
                xy = xy.to(device)

                if is_training:
                    # zero gradients of parameters
                    optimizer.zero_grad()

                # execute model to get outputs
                outputs = model(images)

                # compute MSE loss over x, y coordinates for associated categories
                loss = 0.0
                for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                    loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx+2] - xy[batch_idx])**2)
                loss /= len(category_idx)

                if is_training:
                    # run backpropogation to accumulate gradients
                    loss.backward()

                    # step optimizer to adjust parameters
                    optimizer.step()

                # increment progress
                count = len(category_idx.flatten())
                i += count
                sum_loss += float(loss.item())
                progress_widget.value = i / len(dataset)
                loss_widget.value = sum_loss / i
            
            # エポック終了時に時間を記録してログに出力
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - epoch_start_time
            write_log(f"{epoch_count} Epoch目: {epoch_duration:.2f}秒")
            #get_jetson_nano_memory_usage()
            
            # 最小損失をチェックし、必要に応じてモデルを保存
            if is_training and loss_widget.value < best_loss:
                best_loss = loss_widget.value
                best_loss_form.value = f"{best_loss:.7f}"
                model_dir = os.path.join(asset_root(), 'model')
                os.makedirs(model_dir, exist_ok=True)
                torch.save(model.state_dict(), os.path.join(model_dir, 'best_model.pth'))
                write_log(f"新しいベストモデルが保存されました。Epoch loss: {best_loss:.4f}")

            if is_training:
                epochs_widget.value = epochs_widget.value - 1
            else:
                break

        global_end_time = time.time()
        global_duration = global_end_time - global_start_time
        model_metadata |= {
            "duration": global_duration,
            "end_time": format_epoch(global_end_time),
            "metrics": {
                "best_loss": best_loss,
                "final_loss": loss_widget.value,
            },
            "start_time": format_epoch(global_start_time),
        }
    except Exception as e:
        write_log(f"Error:{e}")

    model = model.eval()

    train_button.disabled = False
    eval_button.disabled = False

train_button.on_click(lambda c: train_eval(is_training=True))
eval_button.on_click(lambda c: train_eval(is_training=False))

def reset_best_loss(change):
    global best_loss
    best_loss = float("inf")
    best_loss_form.value = "inf"
reset_best_loss_button.on_click(reset_best_loss)

train_eval_widget = ipywidgets.VBox([
    epochs_widget,
    progress_widget,
    loss_widget,
    ipywidgets.HBox([best_loss_form, reset_best_loss_button]),
    ipywidgets.HBox([train_button, eval_button]),
])

display(train_eval_widget)

01_find_pwmを実行して、pwmの値を設定してください。

In [ ]:
import Fabo_PCA9685
import time
import smbus
import time
import json

with open('pwm_params.json') as f:
    json_str = json.load(f)

    stop = json_str["pwm_speed"]["stop"]
    left = json_str["pwm_steering"]["left"]
    center = json_str["pwm_steering"]["center"]
    right = json_str["pwm_steering"]["right"]

if stop == 0:
    INITIAL_VALUE=400
else:
    INITIAL_VALUE=stop

In [ ]:
def map_rc(x, in_min, in_max, out_min, out_max):
    return (x - in_min) * (out_max - out_min) // (in_max - in_min) + out_min

def handle(x):
    x = map_rc(x, 224, 0, right, left)

In [ ]:
import functools
import glob
import subprocess
from os.path import join

from fabo import asset_root

def extract_numbers(filename):
    matches = re.findall(r'(\d+)', filename)
    if matches and len(matches) >= 3: 
        return int(matches[-1])  
    else:
        return float('inf') 

def get_file_names(path):
    file_names = os.listdir(path)
    file_names = [os.path.join(path, file_name) for file_name in file_names]
    image_names = []

    image_names = sorted(file_names, key=lambda f: extract_numbers(os.path.basename(f)))
    image_names = [f for f in image_names if os.path.splitext(f)[1].lower() == ".jpg"]
    
    return image_names

@functools.lru_cache(maxsize=1024)
def load_img_from_disk(path):
    return cv2.imread(path, cv2.IMREAD_COLOR) 

# @functools.lru_cache(maxsize=256)
def load_parquet(path):
    return pd.read_parquet(path)

def load_img(no):
    global img, img_filename, load_flag, play_num, running, xy_filenames

    # cv2 colors
    black_color = (0, 0, 0)
    blue_color = (255, 0, 0)
    green_color = (0, 255, 0)
    red_color = (0, 0, 255)

    color_input = SimpleNamespace(cv2=green_color, ipywidgets="green")
    color_inference = SimpleNamespace(cv2=blue_color, ipywidgets="blue")
    color_annotation = SimpleNamespace(cv2=red_color, ipywidgets="red")

    load_task_value = load_task_dropdown.value
    datasets_value = load_datasets_dropdown.value

    xy_path = os.path.join(asset_root(), load_task_value, datasets_value, "xy")
    speed_path = os.path.join(asset_root(), load_task_value, datasets_value, "speed")
    
    t_all = time.perf_counter()
    
    # --- 入力値の読み取り

    pattern = '.*/(\d+)_(\d+)_.*'
    x = 0
    x_read = False
    y = 0
    speed = 0
    speed_read = False
    img = None
    load_ms = -1.0
    try:
        xy_imagenames = []
        if os.path.exists(xy_path):
            xy_imagenames = get_file_names(xy_path)
        if no < len(xy_imagenames):
            xy_name = xy_imagenames[no]
            xy_result = re.match(pattern, xy_name)
            if xy_result:
                x = min(max(int(xy_result.group(1)) + x_annotation_widget.from_input.offset_form.value, 0), IMG_WIDTH)  # 0 <= x <= IMG_WIDTH
                y = min(max(int(xy_result.group(2)), 0), IMG_HEIGHT)  # 0 <= y <= IMG_HEIGHT

                # 現行仕様では x,y = (0,0) が Auto Annotation 未使用によるものか、
                # そうでないかを正確に区別することはできないが、
                # 経験則として Auto Annotation 未使用とみなし、値読込ができない扱いとする。
                if x == 0 and y == 0:
                    x_read = False
                else:
                    x_read = True
        else:
            no_widget.value = len(xy_imagenames) - 1
            write_log("ファイルが存在しません。" + str(len(xy_imagenames)-1) + "以内の値を設定してください。")
            running = False
            return

        speed_imagenames = []
        if os.path.exists(speed_path):
            speed_imagenames = get_file_names(speed_path)
        if no < len(speed_imagenames):
            speed_name = speed_imagenames[no]
            speed_result = re.match(pattern, speed_name)
            if speed_result:
                speed = min(max(int(speed_result.group(2)) + speed_annotation_widget.from_input.offset_form.value, 0), IMG_HEIGHT)  # 0 <= speed <= IMG_HEIGHT
                speed_read = True

        t0 = time.perf_counter()
        img = load_img_from_disk(xy_name)
        load_ms = (time.perf_counter() - t0) * 1000
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")

    if img is None:
        write_log("Image could not be loaded: " + xy_name)
        return
    img_filename = xy_name
    
    # --- 推論値の読み取り
    
    result_x = 0
    result_x_read = False
    result_y = 0
    result_speed = 0
    result_speed_read = False
    infer_ms = -1.0
    try:
        t0 = time.perf_counter()
        preprocessed = preprocess(img)
        output = model(preprocessed).detach().cpu().numpy().flatten()
        infer_ms = (time.perf_counter() - t0) * 1000

        result_x = output[0]
        result_y = output[1]
        result_speed = output[3]

        result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
        result_x = min(max(result_x + x_annotation_widget.from_inference.offset_form.value, 0), IMG_WIDTH)
        result_x_read = True

        result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))

        result_speed = int(IMG_HEIGHT * (result_speed / 2.0 + 0.5))
        result_speed = min(max(result_speed + speed_annotation_widget.from_inference.offset_form.value, 0), IMG_HEIGHT)

        if result_speed> 224:
            result_speed = 224
        elif result_speed < 0:
            result_speed = 0
            
        result_speed_read = True
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")

    # --- アノテーション値の読み取り

    annotation_x = 0
    annotation_x_read = False
    annotation_y = 112  # 非推奨化した y は 112 で固定
    annotation_speed = 0
    annotation_speed_read = False
    annotation_ms = -1.0
    try:
        t0 = time.perf_counter()

        if save_datasets_widget.value:
            dataset_path_p = Path(asset_root()) / save_task_widget.value / save_datasets_widget.value
            parquet_path_p = dataset_path_p / "dataset_v1.parquet"
            if parquet_path_p.exists():
                df = load_parquet(str(parquet_path_p))
                input_image_path = str(Path(img_filename).relative_to(asset_root()))
                df_input_img_matched = df[df["入力画像パス"] == input_image_path]
                if not df_input_img_matched.empty:
                    df_x = df_input_img_matched[df_input_img_matched["値種別"] == "x"]
                    if not df_x.empty:
                        annotation_x = min(max(df_x.iloc[0]["値"] + x_annotation_widget.from_annotation.offset_form.value, 0), IMG_WIDTH)  # 0 <= x <= IMG_WIDTH
                        annotation_x_read = True

                    df_speed = df_input_img_matched[df_input_img_matched["値種別"] == "speed"]
                    if not df_speed.empty:
                        annotation_speed = min(max(df_speed.iloc[0]["値"] + speed_annotation_widget.from_annotation.offset_form.value, 0), IMG_HEIGHT)  # 0 <= speed <= IMG_WIDTH
                        annotation_speed_read = True

        annotation_ms = (time.perf_counter() - t0) * 1000
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")

    # --- 入力値の描画

    speed_bar_thickness = 2
    speed_bar_offset = 2
    t0 = time.perf_counter()

    try:
        marked_img = cv2.copyMakeBorder(
            src=img, top=0, bottom=picture_widget_extra_height, left=0, right=0,
            borderType=cv2.BORDER_CONSTANT, value=(224, 224, 224))
        marked_img = draw_grids(marked_img)

        if x_read:
            marked_img = cv2.circle(marked_img, (int(x), int(y)), 8, color_input.cv2, 3)

        if speed_read:
            marked_img = cv2.line(
                marked_img,
                (0,IMG_HEIGHT+speed_bar_offset+speed_bar_thickness+2),
                (speed,IMG_HEIGHT+speed_bar_offset+speed_bar_thickness+2),
                color_input.cv2,
                speed_bar_thickness)

        _slider = x_annotation_widget.from_input.slider
        _slider.value = x
        if x_read:
            _slider.layout.border = f"solid {color_input.ipywidgets}"
        else:
            _slider.layout.border = f"solid transparent"

        y_widget.value = y

        _slider = speed_annotation_widget.from_input.slider
        _slider.value = speed
        if speed_read:
            _slider.layout.border = f"solid {color_input.ipywidgets}"
        else:
            _slider.layout.border = f"solid transparent"
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")

    # --- 推論値の描画

    try:
        if result_x_read:
            marked_img = cv2.circle(marked_img, (result_x, result_y), 8, color_inference.cv2, 3)

        if result_speed_read:
            marked_img = cv2.line(
                marked_img,
                (0,IMG_HEIGHT+speed_bar_offset),
                (result_speed,IMG_HEIGHT+speed_bar_offset),
                color_inference.cv2,
                speed_bar_thickness)
            # marked_img = cv2.putText(marked_img, "speed:"+str(result_speed), (160,215), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255,255,255))

        _slider = x_annotation_widget.from_inference.slider
        _slider.value = result_x
        if result_x_read:
            _slider.layout.border = f"solid {color_inference.ipywidgets}"
        else:
            _slider.layout.border = f"solid transparent"

        _slider = speed_annotation_widget.from_inference.slider
        _slider.value = result_speed
        if result_speed_read:
            _slider.layout.border = f"solid {color_inference.ipywidgets}"
        else:
            _slider.layout.border = f"solid transparent"
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")

    # --- アノテーション値の描画

    try:
        if annotation_x_read:
            marked_img = cv2.circle(marked_img, (annotation_x, annotation_y), 8, color_annotation.cv2, 3)

        if result_speed_read:
            marked_img = cv2.line(
                marked_img,
                (0,IMG_HEIGHT+speed_bar_offset+2*(speed_bar_thickness+2)),
                (annotation_speed,IMG_HEIGHT+speed_bar_offset+2*(speed_bar_thickness+2)),
                color_annotation.cv2,
                speed_bar_thickness)

        _slider = x_annotation_widget.from_annotation.slider
        _slider.value = annotation_x
        if annotation_x_read:
            _slider.layout.border = f"solid {color_annotation.ipywidgets}"
        else:
            _slider.layout.border = f"solid transparent"

        _slider = speed_annotation_widget.from_annotation.slider
        _slider.value = annotation_speed
        if annotation_speed_read:
            _slider.layout.border = f"solid {color_annotation.ipywidgets}"
        else:
            _slider.layout.border = f"solid transparent"
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")

    draw_ms = (time.perf_counter() - t0) * 1000

    # --- とりまとめ処理

    t0 = time.perf_counter()
    picture_widget.value = bgr8_to_jpeg(marked_img)
    enc_ms = (time.perf_counter() - t0) * 1000
    
    total_ms = (time.perf_counter() - t_all) * 1000
    write_log(
        f"{no} 枚目 {os.path.basename(xy_name)}"
        f" load:{load_ms:.1f}ms"
        f" infer:{infer_ms:.1f}ms"
        f" annotation:{annotation_ms:.1f}ms"
        f" draw:{draw_ms:.1f}ms"
        f" enc:{enc_ms:.1f}ms"
        f" total:{total_ms:.1f}ms"
    )

    if running:
        play_num += 1
        if play_num % 10 == 0:
            write_log(f"{play_num} 回目の再生")
    else:
        next_image_button.disabled = False
        prev_image_button.disabled = False
        write_log(f"{no}枚目の{xy_name}を読込ました。")

def del_pic(c):
    global no, load_task_dropdown,load_datasets_dropdown, xy_filenames
    no = no_widget.value
    name = xy_filenames[no]
    os.remove(name)
    write_log(name + "を削除しました。")
    
def load_dataset(c):
    global img, load_flag
    dataset_path = os.path.join(asset_root(), load_task_dropdown.value, load_datasets_dropdown.value)
    write_log("データセット: " + dataset_path + "を読込みます。")
    load_flag = True
    no = 0
    write_log("初回の読込みには時間がかかります。(30秒〜1分)")
    load_img(no)
    get_jetson_nano_memory_usage()

def next_pic(c):
    global no,x,y,load_flag,skip,check_flag
    load_flag = True
    check_flag = False
    no = no_widget.value
    no = int(no) + skip_dropdown.value
    no_widget.value = no
    next_image_button.disabled = True
    prev_image_button.disabled = True
    load_img(no)
    
def before_pic(c):
    global no,x,y,load_flag,skip,check_flag
    load_flag = True
    check_flag = False
    no = no_widget.value
    no = int(no) - skip_dropdown.value
    if no < 0:
        no = 0
    no_widget.value = no
    next_image_button.disabled = True
    prev_image_button.disabled = True
    load_img(no)

def file_count():
    try:
        xy_path = os.path.join(asset_root(), save_task_widget.value, save_datasets_widget.value, "xy")
        speed_path = os.path.join(asset_root(), save_task_widget.value, save_datasets_widget.value, "speed")
        
        xy_is_dir = os.path.isdir(xy_path)
        
        if xy_is_dir:
            xy_file_count = sum(os.path.isfile(os.path.join(xy_path,name)) for name in os.listdir(xy_path))
            datasets_xy_count_widget.value = xy_file_count
        else:
            datasets_xy_count_widget.value = 0
        
        speed_is_dir = os.path.isdir(speed_path)
        
        if speed_is_dir:
            speed_file_count = sum(os.path.isfile(os.path.join(speed_path,name)) for name in os.listdir(speed_path))
            datasets_speed_count_widget.value = speed_file_count
        else:
            datasets_speed_count_widget.value = 0
            
    except Exception as e:
        #print("An error occurred:", e)
        datasets_xy_count_widget.value = 0
        datasets_speed_count_widget.value = 0


In [ ]:
import datetime
import shutil
from pathlib import Path

from fabo import asset_root

def append_to_parquet(dataset_path, saved_path, input_image_path, value_type, value, remove_saved_path_on_duplicate=True):
    """
    Parquetファイルに行を追加する、または既存レコードを更新する

    Args:
        dataset_path: データセットのパス (例: "dataset/dataset1")
        saved_path: 出力先パス
        input_image_path: 入力画像パス
        value_type: 値種別 ("x", "y", "speed")
        value: 値
    Return:
        更新があれば `True`
    """
    # 妥当性チェック
    def validate_file_paths(**kwargs):
        for k, v in kwargs.items():
            p = Path(v)
            if not p.is_absolute():
                raise ValueError(f"`{k}={v}` は絶対パスである必要があります")
            if not p.exists():
                raise FileNotFoundError(f"`{k}={v}` が存在しません")
            yield p

    dataset_path_p, saved_path_p, input_image_path_p = validate_file_paths(
        dataset_path=dataset_path, saved_path=saved_path, input_image_path=input_image_path)
    parquet_path = str(dataset_path_p / "dataset_v1.parquet")
    saved_path = str(Path(saved_path_p).relative_to(asset_root()))
    input_image_path = str(Path(input_image_path_p).relative_to(asset_root()))

    # ファイル名から InX, InY, InFrameNum をパース
    stem = Path(input_image_path).stem
    in_x, in_y, in_frame_num = None, None, None
    try:
        parts = stem.split('_')
        if len(parts) == 3:
            in_x = int(parts[0])
            in_y = int(parts[1])
            in_frame_num = int(parts[2])
    except (ValueError, IndexError):
        pass  # パースに失敗した場合は None のまま

    # 既存のParquetファイルがあれば読み込み
    if os.path.exists(parquet_path):
        existing_df = pd.read_parquet(parquet_path)

        # 同じ入力画像パス、値種別のレコードを検索
        existing_records = existing_df[
            (existing_df['入力画像パス'] == input_image_path)
            & (existing_df['値種別'] == value_type)
        ]

        if not existing_records.empty:
            # 複数のレコードが存在する場合は警告
            if len(existing_records) > 1:
                write_log(f"⚠️ 警告: 同じ入力画像に対して複数のアノテーションが存在します（{len(existing_records)}件）")
                write_log(f"⚠️ データセットの整合性に問題があります。dataset_v1.parquet を確認し、重複したレコードと画像ファイルを削除してください。")
                write_log(f"⚠️ 入力画像パス: {input_image_path}")

            # 既存レコードが存在する場合
            existing_record = existing_records.iloc[0]
            old_value = existing_record['値']
            old_saved_path = existing_record['出力先パス']
            old_saved_path_p = Path(asset_root()) / old_saved_path

            if not old_saved_path_p.exists():
                write_log(f"Warning: Parquetデータレコードの出力先パスが存在しません: `{old_saved_path_p}` 。入力画像パスから復元します: `{input_image_path_p}`。")
                shutil.copy2(input_image_path_p, old_saved_path_p)
                current_time = datetime.datetime.now(tz=datetime.timezone.utc).timestamp()
                os.utime(old_saved_path_p, (current_time, current_time))

            if old_value != value:
                # 値が異なる場合、出力先パスを再作成する

                # 古い出力先パスを削除する
                old_saved_path_p.unlink()

                # 新しい出力先パスを、入力画像パスからコピーする
                shutil.copy2(input_image_path_p, saved_path_p)
                current_time = datetime.datetime.now(tz=datetime.timezone.utc).timestamp()
                os.utime(saved_path_p, (current_time, current_time))

                # Parquetの該当行を更新
                existing_df.loc[existing_records.index[0], '出力先パス'] = saved_path
                existing_df.loc[existing_records.index[0], '値'] = value
                existing_df.loc[existing_records.index[0], 'UTCタイムスタンプ'] = datetime.datetime.now(timezone.utc).isoformat()

                # Parquetの古い出力先パスを持つ行すべてで、新しい出力先パスを参照するように更新
                existing_df.loc[existing_df['出力先パス'] == old_saved_path, '出力先パス'] = saved_path

                # Parquetファイルに保存
                existing_df.to_parquet(parquet_path, index=False)
                write_log(
                    f"既存アノテーションを更新:"
                    f" saved_path={saved_path_p}"
                    f" {value_type}={old_value} -> {value_type}={value}"
                )
                return True
            else:
                # 同じ場合、特に変更を加えない。
                if remove_saved_path_on_duplicate and old_saved_path_p != saved_path_p:
                    # ただし、古い出力先パスを維持するので、新しい出力先パスは削除する。
                    saved_path_p.unlink(missing_ok=True)
                write_log(
                    f"既に同じ値でアノテーション済み:"
                    f" saved_path={saved_path_p}"
                    f" {value_type}={value}"
                )
                return False
    else:
        existing_df = None

    # 新しい行のデータ
    new_data = {
        '出力先パス': [saved_path],
        '入力画像パス': [input_image_path],
        'InX': [in_x],
        'InY': [in_y],
        'InFrameNum': [in_frame_num],
        '値種別': [value_type],
        '値': [value],
        'UTCタイムスタンプ': [datetime.datetime.now(timezone.utc).isoformat()]
    }
    new_df = pd.DataFrame(new_data)

    # 既存のParquetファイルがあれば追加、なければ新規作成
    if existing_df is not None:
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    # Parquetファイルに保存
    combined_df.to_parquet(parquet_path, index=False)

    return True


In [ ]:
def pop_from_parquet(dataset_path, input_image_path, value_type, keep_saved_path=False):
    """
    Parquetファイルから行を削除する

    Args:
        dataset_path: データセットのパス (例: "dataset/dataset1")
        input_image_path: 入力画像パス
        value_type: 値種別 ("x", "y", "speed")
        keep_saved_path: `False` の場合、出力先パスを削除する。
            データベース間の該当行移行にともなう行削除においては `True` を指定して削除しない運用が必要。
    Return:
        更新があれば削除された行の DataFrame、更新がなければ None
    """
    # 妥当性チェック
    def validate_file_paths(**kwargs):
        for k, v in kwargs.items():
            p = Path(v)
            if not p.is_absolute():
                raise ValueError(f"`{k}={v}` は絶対パスである必要があります")
            if not p.exists():
                raise FileNotFoundError(f"`{k}={v}` が存在しません")
            yield p

    dataset_path_p, input_image_path_p = validate_file_paths(
        dataset_path=dataset_path, input_image_path=input_image_path)
    parquet_path = str(dataset_path_p / "dataset_v1.parquet")
    input_image_path = str(Path(input_image_path_p).relative_to(asset_root()))

    if not os.path.exists(parquet_path):
        # 既存のParquetファイルがなければ処理を切り上げる
        return None

    df = pd.read_parquet(parquet_path)

    # 指定された入力画像パス、値種別に合致するレコードを検索
    matched_df = df[(df['入力画像パス'] == input_image_path)
        & (df['値種別'] == value_type)]

    if matched_df.empty:
        # 該当行がなければ処理を切り上げる
        return None

    # 該当行が存在する場合、削除する
    df = df.drop(matched_df.index)

    if not keep_saved_path:
        # 出力先パスを削除する
        matched_df["出力先パス"].apply(lambda x: (Path(asset_root()) / x).unlink(missing_ok=True))

    # Parquetファイルに保存
    df.to_parquet(parquet_path, index=False)

    return matched_df


In [ ]:
def save_snapshot(_, content, msg):
    global img,x,y,load_flag,save_datasets_widget,save_task_widget
    if content['event'] == 'click' and load_flag == True:
        load_flag = False
        data = content['eventData']
        x = min(max(data['offsetX'], 0), IMG_WIDTH)  # 0 <= x <= IMG_WIDTH
        # y = data['offsetY']
        y = 112  # y の利用が非推奨、かつ可視化の都合、最小・最大の0・224の画像端より中央が見やすいのでニュートラル値に設定

        name = save_datasets_widget.value
        if save_task_widget.value == "":
            write_log("データセット名を指定してください")
        else:
            write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[0] + "カテゴリにデータを追加しました。")
            _dataset = datasets[name]
            saved_path = _dataset.save_entry("xy", img_filename, x, y)

            # Parquetにx, y の2行を追加
            dataset_path = os.path.join(asset_root(), save_task_widget.value, name)
            changed = False
            changed |= append_to_parquet(dataset_path, saved_path, img_filename, "x", x, remove_saved_path_on_duplicate=False)
            changed |= append_to_parquet(dataset_path, saved_path, img_filename, "y", y, remove_saved_path_on_duplicate=True)
            if not changed:
                # append_to_parquet() で変更がない判定の場合、XYDataset.save_entry() による
                # 追加ファイルを削除しているので、XYDataset でファイル数形状を再実行させる。
                _dataset.refresh()

            file_count()

        update_image(None)

def save_x(c, slider):
    x = slider.value
    y = 112  # y の利用が非推奨、かつ可視化の都合、最小・最大の0・224の画像端より中央が見やすいのでニュートラル値に設定

    name = save_datasets_widget.value
    _dataset = datasets[name]
    saved_path = _dataset.save_entry("xy", img_filename, x, y)
    write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[0] + "カテゴリにデータを追加しました。")

    # Parquetにspeedの1行を追加
    dataset_path = os.path.join(asset_root(), save_task_widget.value, name)
    changed = False
    changed |= append_to_parquet(dataset_path, saved_path, img_filename, "x", x, remove_saved_path_on_duplicate=False)
    changed |= append_to_parquet(dataset_path, saved_path, img_filename, "y", y, remove_saved_path_on_duplicate=True)
    if not changed:
        # append_to_parquet() で変更がない判定の場合、XYDataset.save_entry() による
        # 追加ファイルを削除しているので、XYDataset でファイル数形状を再実行させる。
        _dataset.refresh()

    file_count()

    update_image(None)

def save_speed(c, slider):
    speed = slider.value

    name = save_datasets_widget.value
    _dataset = datasets[name]
    saved_path = _dataset.save_entry("speed", img_filename, 0, speed)
    write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[1] + "カテゴリにデータを追加しました。")

    # Parquetにspeedの1行を追加
    dataset_path = os.path.join(asset_root(), save_task_widget.value, name)
    changed = append_to_parquet(dataset_path, saved_path, img_filename, "speed", speed)
    if not changed:
        # append_to_parquet() で変更がない判定の場合、XYDataset.save_entry() による
        # 追加ファイルを削除しているので、XYDataset でファイル数形状を再実行させる。
        _dataset.refresh()

    file_count()

    update_image(None)

In [ ]:

def migrate_x(c, migration_datasets_dropdown):
    src_name = save_datasets_widget.value
    src_dataset_path = os.path.join(asset_root(), save_task_widget.value, src_name)

    removed_df_list = []
    removed_df_list.append(pop_from_parquet(src_dataset_path, img_filename, "x", keep_saved_path=True))
    removed_df_list.append(pop_from_parquet(src_dataset_path, img_filename, "y", keep_saved_path=True))
    removed_df_list = [x for x in removed_df_list if x is not None]
    if len(removed_df_list) == 0:
        return
    if len(removed_df_list) == 1:
        removed_df = removed_df_list[0]
    else:
        removed_df = pd.concat(removed_df_list, ignore_index=True)

    if removed_df is not None:
        dest_name = migration_datasets_dropdown.value
        dest_dataset_path = os.path.join(asset_root(), save_task_widget.value, dest_name)

        changed = False
        def append(row):
            nonlocal changed

            value_type = row["値種別"]
            if value_type in ["x", "y"]:
                dataset_category_dir = "xy"
            elif value_type == "speed":
                dataset_category_dir = "speed"
            else:
                raise ValueError(f"予期せぬ値種別: {value_type}")

            # 出力先パスを、移行元データセットのパスから移行先データセットのパスに調整し、ファイル移動する
            old_saved_path = row["出力先パス"]
            old_saved_path_p = Path(asset_root()) / old_saved_path
            saved_path_p = Path(dest_dataset_path) / dataset_category_dir / old_saved_path_p.name
            if old_saved_path_p.exists():
                # 出力先パスを共有する複数の値種別（e.g. x と y）で、両方が一度に移行される場合、
                # 片方の移行により既存の出力先パスから新しい出力先パスにファイル移動済みのケースが有るため、
                # 存在する場合のみファイル移動する。
                saved_path_p.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(old_saved_path_p, saved_path_p)

            # 移行先に該当行を追加する
            saved_path_abs = str(saved_path_p)
            value = row["値"]
            changed |= append_to_parquet(dest_dataset_path, saved_path_abs, img_filename, value_type, value)

        removed_df.apply(append, axis=1)

        write_log(f"[{save_task_widget.value}/{src_name}] から [{save_task_widget.value}/{dest_name}] に{SAVE_CATEGORIES[0]}カテゴリのデータを移行しました。")

        if not changed:
            # append_to_parquet() で変更がない判定の場合、XYDataset.save_entry() による
            # 追加ファイルを削除しているので、XYDataset でファイル数形状を再実行させる。
            _dataset.refresh()

        file_count()

        update_image(None)

def migrate_speed(c, migration_datasets_dropdown):
    # 移行対象のParquetデータレコードを移行元から取り出し、削除する
    src_name = save_datasets_widget.value
    src_dataset_path = os.path.join(asset_root(), save_task_widget.value, src_name)
    removed_df = pop_from_parquet(src_dataset_path, img_filename, "speed", keep_saved_path=True)

    if removed_df is not None:
        dest_name = migration_datasets_dropdown.value
        dest_dataset_path = os.path.join(asset_root(), save_task_widget.value, dest_name)

        changed = False
        def append(row):
            nonlocal changed

            value_type = row["値種別"]
            if value_type in ["x", "y"]:
                dataset_category_dir = "xy"
            elif value_type == "speed":
                dataset_category_dir = "speed"
            else:
                raise ValueError(f"予期せぬ値種別: {value_type}")

            # 出力先パスを、移行元データセットのパスから移行先データセットのパスに調整し、ファイル移動する
            old_saved_path = row["出力先パス"]
            old_saved_path_p = Path(asset_root()) / old_saved_path
            saved_path_p = Path(dest_dataset_path) / dataset_category_dir / old_saved_path_p.name
            if old_saved_path_p.exists():
                # 出力先パスを共有する複数の値種別（e.g. x と y）で、両方が一度に移行される場合、
                # 片方の移行により既存の出力先パスから新しい出力先パスにファイル移動済みのケースが有るため、
                # 存在する場合のみファイル移動する。
                saved_path_p.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(old_saved_path_p, saved_path_p)

            # 移行先に該当行を追加する
            saved_path_abs = str(saved_path_p)
            value = row["値"]
            changed |= append_to_parquet(dest_dataset_path, saved_path_abs, img_filename, value_type, value)

        removed_df.apply(append, axis=1)

        write_log(f"[{save_task_widget.value}/{src_name}] から [{save_task_widget.value}/{dest_name}] に{SAVE_CATEGORIES[1]}カテゴリのデータを移行しました。")

        if not changed:
            # append_to_parquet() で変更がない判定の場合、XYDataset.save_entry() による
            # 追加ファイルを削除しているので、XYDataset でファイル数形状を再実行させる。
            _dataset.refresh()

        file_count()

        update_image(None)


In [ ]:
from fabo import asset_root

def live():
    global no,running, skip, sleep_time, play_num
    load_flag = True
    play_num = 0
    no = no_widget.value
    while running:
        no += skip
        no_widget.value = no
        try:
            load_img(no)
        except:
            write_log("no: " + no + "のファイルの読込に失敗")
        time.sleep(sleep_time/1000)

def play(c):
    global running, execute_thread, skip, sleep_time, check_flag
    skip = skip_dropdown.value
    sleep_time = sleep_dropdown.value
    running = True
    check_flag = False
    execute_thread = threading.Thread(target=live)
    execute_thread.start()

def stop(c):
    global running, execute_thread, load_flag, check_flag
    running = False
    load_flag = True
    check_flag = False
    try:
        execute_thread.join()
        write_log("STOP")
    except:
        write_log("現在再生されていません。")

def create_dataset(c):
    new_dataset_name = datasets_name_widget.value

    # データセットのディレクトリおよびオブジェクトの作成
    path = os.path.join(asset_root(), save_task_widget.value, new_dataset_name)
    os.makedirs(path)
    datasets[new_dataset_name] = XYDataset(path, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)

    # データセット一覧のウィジェットの更新
    change_save_task(None)
    save_datasets_widget.value = new_dataset_name

    write_log("Datasetを作成しました：" + new_dataset_name)


picture_widget.on_msg(save_snapshot)

# 画像の操作
play_button = ipywidgets.Button(description='▶')
stop_button = ipywidgets.Button(description='⏹')
next_image_button = ipywidgets.Button(description='>')
prev_image_button = ipywidgets.Button(description='<')
load_image_button = ipywidgets.Button(description='読込')
update_image_button = ipywidgets.Button(description='更新')
delete_image_button = ipywidgets.Button(description='削除')

save_model_button = ipywidgets.Button(description='save model')
save_best_model_checkbox = ipywidgets.Checkbox(description='Bestモデルを保存', value=True)

load_model_widget = ipywidgets.Dropdown(options=[],description='読込モデル')
load_model_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
load_model_time_widget = ipywidgets.Text(description='作成日時')
load_datasets_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
save_model_name_widget = ipywidgets.Text(description='保存モデル名',value="model.pth")
save_datasets_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))

dataset_create_button = ipywidgets.Button(description='Create dataset')
datasets_name_widget = ipywidgets.Text(description='Name')

dataset_create_button.on_click(create_dataset)

play_button.on_click(play)
stop_button.on_click(stop)

load_model_button.on_click(load_model)
save_model_button.on_click(save_model)
load_image_button.on_click(load_dataset)
next_image_button.on_click(next_pic)
prev_image_button.on_click(before_pic)
delete_image_button.on_click(del_pic)

load_datasets_dropdown = ipywidgets.Dropdown(options=[], description='dataset')
save_datasets_widget = ipywidgets.Dropdown(options=[], description='dataset')
datasets_xy_count_widget = ipywidgets.IntText(description='XYデータ数')
datasets_speed_count_widget = ipywidgets.IntText(description='速度データ数')

def set_dataset(change):
    datasets[change['new']] = XYDataset(
        os.path.join(asset_root(), save_task_widget.value, change['new']),
        SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
    update_image(None)
save_datasets_widget.observe(set_dataset, names='value')

load_task_dropdown = ipywidgets.Dropdown(options=LOAD_TASK, description='task')
save_task_widget = ipywidgets.Dropdown(options=SAVE_TASK,  value=SAVE_TASK[0], description='task')

x_annotation_widget = AnnotationWidget("x", value_types=["x", "y"], save_func=save_x, migrate_func=migrate_x)
speed_annotation_widget = AnnotationWidget("speed", value_types=["speed"], save_func=save_speed, migrate_func=migrate_speed)

def change_load_task(change):
    path = os.path.join(asset_root(), load_task_dropdown.value)
    try:
        dirs = get_dirs(path)
        load_datasets_dropdown.options = dirs
        if len(load_datasets_dropdown.options) > 0 and load_datasets_dropdown.value not in load_datasets_dropdown.options:
            load_datasets_dropdown.index = 0
    except:
        write_log(path + "が存在していません。")
        load_datasets_dropdown.options = []
load_datasets_refresh_button.on_click(change_load_task)
load_task_dropdown.observe(change_load_task, names='value')
change_load_task(None)

def change_save_task(change):
    path = os.path.join(asset_root(), save_task_widget.value)
    try:
        os.makedirs(path, exist_ok=True)
        dirs = get_dirs(path)
        save_datasets_widget.options = dirs
        # 保存先データセットは明示的に指定させる
        # if len(save_datasets_widget.options) > 0 and save_datasets_widget.value not in save_datasets_widget.options:
        #     save_datasets_widget.index = 0
    except:
        write_log(path + "が存在していません。")
        save_datasets_widget.options = ['']
save_datasets_refresh_button.on_click(change_save_task)
save_task_widget.observe(change_save_task, names='value')
change_save_task(None)

def change_save_dataset(change):
    file_count()
save_datasets_widget.observe(change_save_dataset, names='value')
change_save_dataset(None)

def change_sleep(change):
    global sleep_time
    sleep_time = sleep_dropdown.value
sleep_dropdown.observe(change_sleep, names='value')

def change_skip(change):
    skip = skip_dropdown.value
skip_dropdown.observe(change_skip, names='value')

def model_list(change):
    try:
        files = glob.glob(os.path.join(asset_root(), 'model', '*.pth'), recursive=True)
        files.insert(0,"[new]")
        load_model_widget.options = files
        if len(load_model_widget.options) > 0 and load_model_widget.value not in load_model_widget.options:
            load_model_widget.index = 0
    except:
        load_model_widget.options = []
load_model_refresh_button.on_click(model_list)
model_list(None)

def change_file(change):
    try:
        file = load_model_widget.value
        ts = os.path.getctime(file)
        d = datetime.datetime.fromtimestamp(ts)
        s = d.strftime('%Y-%m-%d %H:%M:%S')
        load_model_time_widget.value = s
    except:
        load_model_time_widget.value = ""
load_model_widget.observe(change_file, names='value')

def update_image(change):
    global load_flag, no
    load_flag = True
    no = no_widget.value
    load_img(no)
update_image_button.on_click(update_image)


In [ ]:
import numpy as np
from functools import partial

WIDTH = 80
HEIGHT = 80
SIZE = 8

check_image_button = ipywidgets.Button(description=f'{SIZE}個単位チェック')
check_next_images_button = ipywidgets.Button(description=f'[{SIZE}]>')
check_prev_images_button = ipywidgets.Button(description=f'<[-{SIZE}]')
check_start_index_widget = ipywidgets.IntText(description='開始位置')
check_end_index_widget = ipywidgets.IntText(description='終了位置')
check_image_count_widget = ipywidgets.IntText(description='最終画像位置')
check_update_button = ipywidgets.Button(description='更新')

# 画像を表示するウィジェット
snapshot_widgets = []
snapshot_button_widgets = []


def edit_image(index, b):
    global load_flag,no,check_flag
    no = check_no + index
    load_flag = True
    check_flag = False
    load_img(no)
    no_widget.value = no
    
for i in range(SIZE):
    image = ipywidgets.Image(width=WIDTH, height=HEIGHT)
    edit_button = ipywidgets.Button(description="編集", layout=ipywidgets.Layout(width=f'{WIDTH}px', height=f'30px'))
    edit_button.on_click(partial(edit_image, i))
    black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    image.value = bgr8_to_jpeg(black_image)
    snapshot_widgets.append(image)
    snapshot_button_widgets.append(VBox([edit_button,image]))

def get_load_dataset_length():
    xy_path = os.path.join(asset_root(), load_task_dropdown.value, load_datasets_dropdown.value, "xy")
    xy_filenames = get_file_names(xy_path)
    check_image_count_widget.value = len(xy_filenames)
    last_no = len(xy_filenames)
    return last_no

def next_images(c):
    global check_no,last_no,check_flag
    check_next_images_button.disabled = True
    check_prev_images_button.disabled = True
    write_log(f"check_flag:"+str(check_flag))
    if check_flag == False:
        check_no = no_widget.value
        check_flag = True
    else:
        check_no += SIZE
        write_log(f"check_no: {check_no}")
    try:
        last_no = get_load_dataset_length()
        if check_no < last_no:
            load_images(check_no)
        else:
            check_next_images_button.disabled = False
            check_prev_images_button.disabled = False
    except:
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    
def prev_images(c):
    global check_no,check_flag
    check_next_images_button.disabled = True
    check_prev_images_button.disabled = True
    write_log(f"check_flag:"+str(check_flag))
    if check_flag == False:
        check_no = no_widget.value
        check_no -= SIZE
        check_flag = True
    else:
        check_no -= SIZE
        write_log(f"check_no: {check_no}")
    try:
        last_no = get_load_dataset_length()
        if check_no < 0:
            check_no = 0
        load_images(check_no)
    except:
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    
def load_images(c):
    global check_no, last_no, snapshot_widgets
    write_log("画像を" + str(SIZE) + "枚読込み、推論結果を付与します。")
    try:
        xy_path = os.path.join(asset_root(), load_task_dropdown.value, load_datasets_dropdown.value, "xy")
        xy_filenames = get_file_names(xy_path)
        last_no = len(xy_filenames)
        check_image_count_widget.value = last_no
        now_no = 0
        write_log(f"{xy_path}のデータセットを読み込みます。データ数(xy): {last_no}")
        for i in range(SIZE):
            now_no = check_no + i
            if now_no < last_no:
                try:
                    xy_name = xy_filenames[now_no]
                    img = cv2.imread(xy_name)
                    preprocessed = preprocess(img)
                    output = model(preprocessed).detach().cpu().numpy().flatten()
                    result_x = output[0]
                    result_y = output[1]
                    result_speed = output[3]
                    result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
                    result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))
                    result_speed = int(IMG_HEIGHT * (result_speed / 2.0 + 0.5))
                    marked_img = cv2.circle(img, (int(result_x), int(result_y)), 8, (255, 0, 0), 3)
                    marked_img = cv2.line(marked_img,(219,224-result_speed),(219,224),(0,140,255),3)
                    marked_img = cv2.putText(marked_img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))

                    snapshot_widgets[i].value = bgr8_to_jpeg(marked_img)
                    
                    time.sleep(10/1000)
                except Exception as e:
                    write_log(f"{e}")
                    black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
                    snapshot_widgets[i].value = bgr8_to_jpeg(black_image)
            else:
                black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
                snapshot_widgets[i].value = bgr8_to_jpeg(black_image)
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    except Exception as e:
        write_log(f"{e}")
    #get_jetson_nano_memory_usage()

def update_images(c):
    global check_no
    check_no = check_start_index_widget.value
    load_images(check_no)
    
check_no = 0
check_image_button.on_click(load_images)
check_prev_images_button.on_click(prev_images)
check_next_images_button.on_click(next_images)
check_update_button.on_click(update_images)

In [ ]:
movie_button = ipywidgets.Button(description='動画の作成')
movie_name_widget = ipywidgets.Text(description='動画名',value="run_video")

def make_movie(change):
    global model
    
    if not movie_name_widget.value.strip():
        write_log("ファイル名を指定してください。")
        return 
    write_log("動画を作成します。")
    path = os.path.join(asset_root(), "video")
    os.makedirs(path, exist_ok=True)
    output = os.path.join(path, movie_name_widget.value + ".mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(30 / movie_skip_dropdown.value)
    outfh = cv2.VideoWriter(output, fourcc, fps, (224, 224))

    file_list = sorted(
        glob.glob(os.path.join(asset_root(), load_task_dropdown.value, load_datasets_dropdown.value, "xy", "*.jpg")),
        key=lambda f: extract_numbers(os.path.basename(f)),
    )

    try:
        res_num = len(file_list)
        
        count = 0
        skip_movie = movie_skip_dropdown.value
        terminal_time = 1 / (30 / skip_movie)
        current_time = 0
        process_time = 0
        total_process_time = 0
        for i, file_name in enumerate(file_list):
            if i % skip_movie == 0:
                current_time += terminal_time
                img = cv2.imread(file_name)

                process_time = time.time()
                preprocessed = preprocess(img)
                output = model(preprocessed).detach().cpu().numpy().flatten()

                img = cv2.putText(img, f"I={i}", (10, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                # XY
                result_x = float(output[0])
                result_y = float(output[1])
                result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
                result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))
                img = cv2.circle(img, (int(result_x), int(result_y)), 8, (255, 0, 0), 3)

                # Speed
                result_speed = output[3]
                result_speed = int(IMG_WIDTH * (result_speed / 2.0 + 0.5))
                if result_speed > 224:
                    result_speed = 244
                elif result_speed < 0:
                    result_speed = 0
                img = cv2.line(img, (218, 0), (218, 224), (0, 0, 0), 5)
                img = cv2.line(img, (219, 224 - result_speed), (219, 224), (0, 140, 255), 3)
                img = cv2.putText(img, "speed:" + str(result_speed), (160, 215), cv2.FONT_HERSHEY_SIMPLEX, 0.3,
                                  (255, 255, 255))
                total_process_time += time.time() - process_time

                if i % (skip_movie * 10) == 0:
                    write_log(
                        f"{current_time:.1f}秒まで完了"
                        f", 推論平均: {total_process_time / 10 * 1000:.1f}ms"
                        f", {int(i / skip_movie)}枚目/{int(res_num / skip_movie)}枚中を処理中")
                    total_process_time = 0
                outfh.write(img)
                time.sleep(5/1000)
                del img
    except Exception as e:
        write_log(f"Error:{e}")
    finally:
        # エラーが発生しても確実にリソースを解放する
        outfh.release()
        write_log("動画の出力が完了しました。")
        get_jetson_nano_memory_usage()

movie_button.on_click(make_movie)

In [ ]:
import subprocess
import re

used_memory_widget = ipywidgets.IntText(description='Useメモリ', value=1)
total_memory_widget = ipywidgets.IntText(description='全メモリ', value=1)
memory_button = ipywidgets.Button(description='使用メモリ量の取得')

def get_jetson_nano_memory_usage(event=None):
    command = 'tegrastats'
    try:
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        
        mem_usage_pattern = re.compile(r'RAM (\d+)/(\d+)MB')
        
        max_lines_to_read = 10
        for _ in range(max_lines_to_read):
            line = process.stdout.readline()
            if not line:
                break 
            matches = mem_usage_pattern.search(line)
            if matches:
                used_memory_widget.value = int(matches.group(1))
                total_memory_widget.value = int(matches.group(2))
                write_log("使用メモリ： " + str(used_memory_widget.value) + "/" + str(total_memory_widget.value))
                process.kill()
                return
        
        process.kill()  
        return

    except subprocess.CalledProcessError as e:
        return

get_jetson_nano_memory_usage()
memory_button.on_click(get_jetson_nano_memory_usage)

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.使用する推論モデル】</b> [New]は新規モデル。')
title2 = ipywidgets.HTML('<b>【2.読込元データセット】</b> アノテーションを実施するデータセットを選択。')
title3 = ipywidgets.HTML('<b>【3.保存先データセット】</b> データセットの保存先を選択。')
title4 = ipywidgets.HTML('<b>【4.アノテーションの実施】</b> 緑◯がアノテーション, 青◯がAIでの推論。車両の走らせたい場所で、画面をクリックすると保存先データセットのxyにデータが登録されます。Speedは[速度追加]で追加します。')
title5 = ipywidgets.HTML('<b>【5.学習】</b> EPOCH指定で学習できます。')
title6 = ipywidgets.HTML('<b>【6.評価動画の作成】</b> 動画を作成します。')
vspacer = ipywidgets.HTML(value="", layout=ipywidgets.Layout(width='auto', flex_grow=1, border='none'))

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([load_model_widget, load_model_refresh_button, load_model_time_widget, load_model_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title2,
    ipywidgets.HBox([load_datasets_dropdown, load_datasets_refresh_button, load_task_dropdown, load_image_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title3,
    ipywidgets.HBox([save_datasets_widget, save_datasets_refresh_button, save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([Label('datasetの新規作成'),datasets_name_widget,dataset_create_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title4,
    ipywidgets.HBox([play_button, stop_button, prev_image_button, next_image_button]),
    ipywidgets.HBox([sleep_dropdown, skip_dropdown]),
    ipywidgets.HBox([no_widget, update_image_button, delete_image_button]),
    ipywidgets.HBox([picture_widget, speed_annotation_widget.widget, x_annotation_widget.widget]),
    ipywidgets.HBox([Label(f'{SIZE}個単位での処理'), check_prev_images_button, check_next_images_button]),
    ipywidgets.HBox(snapshot_button_widgets),
    ipywidgets.HBox([save_datasets_widget, save_datasets_refresh_button, save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget, datasets_speed_count_widget]),
    ipywidgets.HBox([used_memory_widget, total_memory_widget, memory_button]),
    process_widget,
    separator,
    title5,
    ipywidgets.HBox([save_datasets_widget, save_datasets_refresh_button, save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([epochs_widget,train_button,eval_button]),
    ipywidgets.HBox([progress_widget,loss_widget]),
    ipywidgets.HBox([save_model_name_widget, save_model_button, save_best_model_checkbox]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title6,
    ipywidgets.HBox([load_datasets_dropdown, load_datasets_refresh_button, load_task_dropdown]),
    ipywidgets.HBox([movie_name_widget,movie_skip_dropdown,movie_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
])
display(data_collection_widget)